In [25]:
import random
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold



In [3]:
def add_numeric_index(df):
    df['numeric_id'] = range(1, len(df) + 1)
    return df


In [ ]:
rdb = add_numeric_index( pd.read_csv('bak_v1/Datasets/clean_db_reddb.csv') )
nfa = add_numeric_index( pd.read_csv('bak_v1/Datasets/clean_db_nfa.csv') )
qm9 = add_numeric_index( pd.read_csv('bak_v1/Datasets/clean_db_qm9.csv') )

In [ ]:
desc_rdb = add_numeric_index( pd.read_csv('bak_v1/Datasets/descNormal_clean_db_reddb.csv') )
desc_nfa = add_numeric_index( pd.read_csv('bak_v1/Datasets/descNormal_clean_db_nfa.csv') )
desc_qm9 = add_numeric_index( pd.read_csv('bak_v1/Datasets/descNormal_clean_db_qm9.csv') )

In [6]:
rdb.head()

,smiles,homo,lumo,gap,numeric_id
0,C1C(O)=C(C(=O)O)C=C1O,-0.17872,-0.08681,0.09191,1
1,C1C(O)=C(C(=O)O)C(=C1O)C(=O)O,-0.20035,-0.10352,0.09683,2
2,C1C(O)=C(F)C=C1O,-0.16346,-0.04380,0.11966,3
3,C1C(O)=C(F)C(F)=C1O,-0.16777,-0.04303,0.12474,4
4,C1C(O)=C(N)C(N)=C1O,-0.12528,-0.03079,0.09449,5


In [7]:
desc_rdb.head()

,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,MolWt,FpDensityMorgan1,BalabanJ,HallKierAlpha,Ipc,PEOE_VSA1,...,NHOHCount,NumAromaticCarbocycles,NumAromaticHeterocycles,NumAromaticRings,NumHAcceptors,NumHDonors,NumRotatableBonds,RingCount,MolLogP,numeric_id
0,0.540137,0.040874,0.735976,0.610715,0.066761,0.770764,0.499418,0.839557,4.654125e-09,0.231080,...,0.055556,0.0,0.0,0.0,0.05,0.1,0.111111,0.0,0.543144,1
1,0.567951,0.265893,0.690902,0.589178,0.118307,0.418094,0.621136,0.766252,2.020850e-08,0.308107,...,0.111111,0.0,0.0,0.0,0.10,0.2,0.222222,0.0,0.498493,2
2,0.728807,0.038363,0.798759,0.610746,0.036286,0.770764,0.448178,0.903181,1.048807e-09,0.154054,...,0.000000,0.0,0.0,0.0,0.00,0.0,0.000000,0.0,0.612136,3
3,0.732740,0.260871,0.718781,0.648280,0.057357,0.465116,0.516549,0.893499,2.397208e-09,0.154054,...,0.000000,0.0,0.0,0.0,0.00,0.0,0.000000,0.0,0.636476,4
4,0.394398,0.030691,0.886795,0.413774,0.050388,0.465116,0.516549,0.857538,2.397208e-09,0.327027,...,0.222222,0.0,0.0,0.0,0.10,0.2,0.000000,0.0,0.470910,5


In [8]:
desc_rdb.shape, rdb.shape

((15238, 74), (15238, 5))

In [9]:
def filter_dataframe_by_ids(dataframe, id_column, id_list):
    """Filter a dataframe by keeping only the rows whose identifier is in a given list.

    This function selects rows from the input dataframe where the values in the
    specified identifier column match any value contained in the external list of ids.

    Args:
        dataframe: Input pandas dataframe that contains the identifiers.
        id_column: Name of the column in the dataframe that stores the identifiers.
        id_list: List of integer identifiers to be used for filtering.

    Returns:
        filtered_df: A pandas dataframe containing only the rows whose identifier
            matches one of the values in the provided list.
    """
    # Use pandas .isin() to check if each value of the identifier column
    # is present in the external list of identifiers
    mask = dataframe[id_column].isin(id_list)

    # Apply the mask to filter the dataframe
    filtered_df = dataframe[mask].reset_index(drop=True)

    return filtered_df


In [10]:

def scaffold_split_dataframe(dataframe, descriptors, smiles_column, gap_column, seed=42):
    """Split a dataframe of molecules into train, validation, and test sets using scaffold-based splitting.

    The splitting is performed by grouping molecules that share the same Bemis-Murcko scaffold,
    ensuring that structurally similar compounds do not end up in both training and testing sets.

    Args:
        dataframe: Input pandas dataframe containing at least SMILES and gap columns.
        descritptors: Input pandas dataframe containing descriptors in dataframe
        smiles_column: Name of the column in the dataframe containing SMILES strings.
        gap_column: Name of the column in the dataframe containing gap values.
        seed: Random seed to ensure reproducibility.

    Returns:
        train_df: Pandas dataframe containing 40% of molecules (training set).
        valid_df: Pandas dataframe containing 10% of molecules (validation set).
        tests_df: Pandas dataframe containing 50% of molecules (test set).

        desc_train_df: Pandas dataframe containing 40% of corresponding descriptors in train_df
        desc_valid_df: Pandas dataframe containing 10% of corresponding descriptors in valid_df
        desc_tests_df: Pandas dataframe containing 50% of corresponding descriptors in tests_df

    """
    random.seed(seed)

    # Dictionary: scaffold string -> list of dataframe indices
    scaffold_to_indices = {}

    for idx, smiles in enumerate(dataframe[smiles_column]):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue  # Skip invalid SMILES
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        if scaffold not in scaffold_to_indices:
            scaffold_to_indices[scaffold] = []
        scaffold_to_indices[scaffold].append(idx)

    # Sort scaffolds by size of their molecule groups (largest first)
    scaffold_groups = sorted(scaffold_to_indices.values(), key=lambda x: len(x), reverse=True)

    # Shuffle groups to randomize distribution
    random.shuffle(scaffold_groups)

    # Flatten scaffold groups into one index list
    all_indices = [idx for group in scaffold_groups for idx in group]

    # Calculate split sizes
    total_size = len(all_indices)
    train_size = int(0.5 * total_size)
    

    # Assign indices to splits
    train_indices = all_indices[ :train_size ]
    tests_indices = all_indices[ train_size: ]

    # Create dataframes
    train_df = dataframe.iloc[train_indices].reset_index(drop=True)
    tests_df = dataframe.iloc[tests_indices].reset_index(drop=True)
    print('Train Shape', train_df.shape)
    print('Tests Shape', tests_df.shape)


    desc_train_df = filter_dataframe_by_ids(descriptors, 'numeric_id', train_df['numeric_id'].tolist() )
    desc_tests_df = filter_dataframe_by_ids(descriptors, 'numeric_id', tests_df['numeric_id'].tolist() )

    train_df = train_df.drop( columns = ['numeric_id'])
    tests_df = tests_df.drop( columns = ['numeric_id'])

    desc_train_df = desc_train_df.drop( columns = ['numeric_id'])
    desc_tests_df = desc_tests_df.drop( columns = ['numeric_id'])


    return train_df, tests_df, desc_train_df, desc_tests_df


In [11]:

output_rdb = scaffold_split_dataframe(rdb, desc_rdb, 'smiles', 'gap', seed=42)
rdb_train, rdb_tests, desc_rdb_train, desc_rdb_tests = output_rdb

Train Shape (7619, 5)
Tests Shape (7619, 5)


In [12]:
output_nfa = scaffold_split_dataframe(nfa, desc_nfa, 'smiles', 'gap', seed=42)
nfa_train, nfa_tests, desc_nfa_train,  desc_nfa_tests = output_nfa

Train Shape (25624, 5)
Tests Shape (25624, 5)


In [13]:
output_qm9 = scaffold_split_dataframe(qm9, desc_qm9, 'smiles', 'gap', seed=42)
qm9_train, qm9_tests, desc_qm9_train, desc_qm9_tests = output_qm9

Train Shape (66081, 5)
Tests Shape (66082, 5)


In [14]:
rdb_train.head()

,smiles,homo,lumo,gap
0,c1cc(C(=O)O)c(C(=O)O)c(c12)c(O)c([nH]2)O,-0.17918,-0.11910,0.06008
1,O=C(O)c(c1)cc(C(=O)O)c(c12)c(O)c([nH]2)O,-0.18249,-0.10604,0.07645
2,O=C(O)c1ccc(C(=O)O)c(c12)c(O)c([nH]2)O,-0.19472,-0.12220,0.07252
3,O=C(O)c(c1)c(C(=O)O)cc(c12)c(O)c([nH]2)O,-0.19196,-0.10983,0.08213
4,O=C(O)c1cc(C(=O)O)cc(c12)c(O)c([nH]2)O,-0.18731,-0.10507,0.08224


In [ ]:


def count_unique_scaffolds(dataframe, smiles_column):
    """Count the number of unique Bemis–Murcko scaffolds in a dataframe.

    This function extracts Bemis–Murcko scaffolds from SMILES strings in the dataframe
    and calculates how many unique scaffolds are present.

    Args:
        dataframe: Input pandas dataframe that contains molecules in SMILES format.
        smiles_column: Name of the column that stores SMILES strings.

    Returns:
        unique_count: Integer with the number of unique scaffolds.
    """
    scaffolds = []
    for smiles in dataframe[smiles_column]:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue  # Skip invalid SMILES
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
        scaffolds.append(scaffold)

    unique_count = len(set(scaffolds))
    return unique_count


In [16]:
n_scaffolds_train = count_unique_scaffolds(rdb_train, "smiles")
n_scaffolds_tests = count_unique_scaffolds(rdb_tests, "smiles")
print(f"Number of unique scaffolds in train set: {n_scaffolds_train }")
print(f"Number of unique scaffolds in train set: {n_scaffolds_tests }")


Number of unique scaffolds in train set: 28
Number of unique scaffolds in train set: 22


In [17]:
n_scaffolds_train = count_unique_scaffolds(nfa_train, "smiles")
n_scaffolds_tests = count_unique_scaffolds(nfa_tests, "smiles")
print(f"Number of unique scaffolds in train set: {n_scaffolds_train }")
print(f"Number of unique scaffolds in train set: {n_scaffolds_tests }")


Number of unique scaffolds in train set: 17804
Number of unique scaffolds in train set: 17998


In [18]:
n_scaffolds_train = count_unique_scaffolds(qm9_train, "smiles")
n_scaffolds_tests = count_unique_scaffolds(qm9_tests, "smiles")
print(f"Number of unique scaffolds in train set: {n_scaffolds_train }")
print(f"Number of unique scaffolds in train set: {n_scaffolds_tests }")


Number of unique scaffolds in train set: 7682
Number of unique scaffolds in train set: 8134


In [19]:
5296 + 2387, 8134

(7683, 8134)

In [20]:
# !pwd

In [21]:
rdb_train.to_csv('scaffold_splitting/rdb_train.csv')
rdb_tests.to_csv('scaffold_splitting/rdb_tests.csv')
desc_rdb_train.to_csv('scaffold_splitting/desc_rdb_train.csv')
desc_rdb_tests.to_csv('scaffold_splitting/desc_rdb_tests.csv')

In [22]:
nfa_train.to_csv('scaffold_splitting/nfa_train.csv')
nfa_tests.to_csv('scaffold_splitting/nfa_tests.csv')
desc_nfa_train.to_csv('scaffold_splitting/desc_nfa_train.csv')
desc_nfa_tests.to_csv('scaffold_splitting/desc_nfa_tests.csv')

In [23]:
qm9_train.to_csv('scaffold_splitting/qm9_train.csv')
qm9_tests.to_csv('scaffold_splitting/qm9_tests.csv')
desc_qm9_train.to_csv('scaffold_splitting/desc_qm9_train.csv')
desc_qm9_tests.to_csv('scaffold_splitting/desc_qm9_tests.csv')

In [ ]:
# Train set will be used to perform CrossValidation with k-fold = 5
# Tests set will be used after training and location of best hyperparameters